In [18]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Imports

In [ ]:
# general imports
import scipy
from scipy.signal import hilbert
import os
import yaml
import mne
import pandas as pd
import numpy as np
import yasa
import sys

# import from custom script
import shared_processing_functions as spf

# import from different directory
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from hdf5_files import Artefacts_Detection as ad

# Functions

### Finding .edf files in input eeg path

In [20]:
def find_edf(path):
    raw_files = []
    for file in os.listdir(path):
        if not file.startswith('.'):
            if ".edf" in file:
                raw_files.append(os.path.join(path, file))
    
    return raw_files

### Create new directory for subject

In [21]:
def create_subject_directory(dir_path):
    # try to create new directory for path
    if os.path.exists(dir_path):
        print("Directory already exists.")
        # check if the directory contains a minimum of 3 files
        print("Checking to see if directory has a minimum of 3 files.")
        if len(os.listdir(dir_path)) >= 3:
            print("Directory has more than the minimum of 3 files.\n\
                  Data has most likely been properly extracted, \
                  manually confirm to make sure.")
            empty_dir = False
        # remove directory because of incomplete data extraction
        else:
            print("Removing files from directory, because data has \
                  not been properly extracted.")
            # iterating through files and removing them
            for file in os.listdir(dir_path):
                os.remove(os.path.join(dir_path, file))
            print("Old files have been removed.")
            empty_dir = True
    else:
        os.mkdir(dir_path)
        print("New directory has succesfully been created.")
        empty_dir = True

    return empty_dir

### Add annotation to raw subject

In [22]:
def add_annotation(anno_file, raw):
    # load mat file with annotation
    mat_data = scipy.io.loadmat(anno_file)
    states = mat_data["states"]

    # get values from 2d array
    descriptions = [str(int(s[0])) for s in states]  # state labels as strings
    onsets = [float(s[2]) for s in states]           # onset in seconds
    durations = [float(s[3]) for s in states]        # duration in seconds

    # create annotations object
    annotations = mne.Annotations(onset=onsets, duration=durations, description=descriptions)

    # set annotations
    raw.set_annotations(annotations)

### Crop the raw data to the size of the annotation

In [23]:
def crop_to_anno(raw):
    segments = []
    for onset, duration in zip(raw.annotations.onset, raw.annotations.duration):
        seg = raw.copy().crop(tmin=onset, tmax=min(onset+duration, raw.times[-1]))  # lazy if preload=False
        segments.append(seg)

    raw_cropped = mne.concatenate_raws(segments)
    
    return raw_cropped

### Get the powerband DataFrame

In [24]:
def get_power_band(raw):
    sf = raw.info['sfreq']
    results = []
    raw.get_channel_types()
    raw_eeg = raw.copy().pick_types(eeg=True, eog=False, emg=False, misc=False)
    raw_eeg.get_channel_types()
    print(raw_eeg.ch_names)
    # Loop over annotations
    for annot in raw_eeg.annotations:
        stage = annot['description']
        onset = annot['onset']
        duration = annot['duration']

        n_chunks = int(np.ceil(duration / chunk_duration))

        for i in range(n_chunks):
            tmin = onset + i * chunk_duration
            tmax = min(onset + duration, tmin + chunk_duration)

            # Crop raw to this chunk
            raw_chunk = raw_eeg.copy().crop(tmin=tmin, tmax=min(tmax, raw_eeg.times[-1]))
            data = raw_chunk.get_data()           # channels x samples
            data = data.astype(float)             # convert to float for YASA
            if data.shape[1] < 1000:
                continue

            # Compute bandpower using YASA (returns DataFrame)
            bp_df = yasa.bandpower(data, sf=sf, bands=bands)

            # Add the stage column
            bp_df['Stage'] = stage

            # Add channel names
            bp_df['Channel'] = raw_eeg.info['ch_names']

            # Append to results
            results.append(bp_df)

    # Concatenate all chunks
    results_df = pd.concat(results, ignore_index=True)

    # Optional: average per stage/channel/band
    bp_mean = results_df.groupby(['Stage', 'Channel']).mean().reset_index()

    return bp_mean

### Find the electrode with the highest power and create bipolar channel with reference

In [25]:
def best_channel(bp, stage, band):
    # get dataframe with only target stage electrodes
    df_stage = bp[bp['Stage'] == stage]
    # get electrode with highest power
    max_channel = df_stage.loc[df_stage[band].idxmax()]['Channel']

    return max_channel

### Some processing

In [26]:
def process_channels(raw_obj, high_pass, low_pass, picks):
    # instead of directly altering the data, use this function to go 
    # through all channels, lambda function was gotten from chatGPT
    raw_obj.apply_function(lambda x: mne.filter.detrend(x, axis=0, order=1), picks='all')
    # bandpass
    raw_obj.filter(l_freq=high_pass, h_freq=low_pass, picks=picks)

### Create bipolar channel

In [27]:
def create_bipolar(raw_obj, picks, band):
    # create bipolar channel
    mne.set_bipolar_reference(
        raw_obj, 
        anode=picks[0], 
        cathode=picks[0], 
        ch_name=band, 
        copy=False
    )

### Custom filters and channel creation for EMG electrodes

In [28]:
def get_emg(raw_cropped):
    data_copy = raw_cropped.copy()
    emg_data = data_copy.pick(['E240', 'E243'])
    emg_data.load_data()

    emg_data._data = mne.filter.detrend(emg_data._data, axis=1, order=1)

    emg_data.notch_filter(freqs=[50, 100])
    #data.filter(l_freq=0.25, h_freq=40, picks=['EMG1', 'EMG2'])
    emg_data.filter(l_freq=10, h_freq=40)
    #picks = mne.pick_channels(data.ch_names, ['E240', 'E243'])
    #picks = mne.pick_channels(data.ch_names, ['EMG1', 'EMG2'])
    # data_c = data.get_data(picks).copy()
    emg_data = emg_data.get_data()
    l, r = emg_data
    l, r = np.abs(l), np.abs(r)

    # # Step 3: envelope via Hilbert
    l = np.abs(hilbert(l))
    r = np.abs(hilbert(r))

    # Step 4: combine (average)
    emg_combined = (l + r) / 2

    return emg_combined

In [29]:
def create_average_channel(raw_obj, indices, name):
    # take average of reference channels and create a new ref channel
    avg_ch = mne.channels.combine_channels(
        raw_obj, 
        groups={name: indices},
        method="mean"
    )

    return avg_ch

# Setup

### Access config parameters

In [30]:
with open('extract_egi_config.yaml') as p:
    params = yaml.safe_load(p)

### Variables

In [31]:
# wake time (s) to save before first sleep and after last sleep
# (30 mins, so 30(s) * 60(s))
wake_time = params['variables']['wake_time']
# stage names and corresponding ids
bands_list = params['variables']['power_bands']
bands = [tuple(band_list) for band_list in bands_list]
# chunk duration for psd calculation (helps with memory issues)
chunk_duration = params['variables']['chunk_time']
# channel and their corresponding names
channel_types = params['variables']['channel_types']
# stages and their corresponding indexes
stage_index = params['variables']['stage_index']
# target bands to isolate
target_bands = params['variables']['target_bands']
# which stage to look at for powerband extraction
target_stages = params['variables']['target_stages']
emg_ch = params['variables']['emg_ch']
eog_ch = params['variables']['eog_ch']
ref_ch = params['variables']['ref_ch']

### Paths

In [32]:
# path to general data
path_to_data = params['paths']['data']
# partial path to edf files
path_to_edf = params['paths']['edf']
# partial path to raw hypnogram annotation files
path_to_anno = params['paths']['annotation']
# partial path to the output for the .mat files
path_to_output = params['paths']['output']

# complete file paths
# edf_path = os.path.join(path_to_data, path_to_edf)
# anno_path = os.path.join(path_to_data, path_to_anno)
#output_path = os.path.join(path_to_data, path_to_output)
edf_path = path_to_edf
anno_path = path_to_anno
output_path = path_to_output

# Main

### Find all edf files

In [33]:
# find all edf files
edf_files = find_edf(edf_path)

print(f"Amount of files available: {len(edf_files)}")

Amount of files available: 66


### Create mat files for each subject

In [ ]:
for edf_file_path in edf_files:
    # dictionary to save the different bands with highest 
    # power in specific channel
    channel_bands = {}
    # create directory to save info to
    final_results = {}


    # isolate subject name from path
    subject = edf_file_path.split('\\')[-1].split('_', 1)[1].split('.')[0]
    output_list = list(os.listdir(output_path))
    if f'{subject}_Fpz-Cz' in output_list and f'{subject}_Pz-Oz':
    print(f'Extracting data from subject: {subject}')
    print('-' * 50)

    # isolate subject name from path
    file = edf_file_path.split('\\')[-1]
    subject = file.split('_', 1)[1].split('.')[0]
    print(f'Extracting data from subject: {subject}')
    print('-' * 50)
    
    # # new output path for subject
    # subject_output_path = os.path.join(output_path, subject)
    # # check existance of dir and wether data has already been extracted
    # if create_subject_directory(subject_output_path):
    #     pass
    # else:
    #     continue
    
    ### loading data
    # load edf into mne object
    raw = mne.io.read_raw_edf(edf_file_path, verbose='error')

    # raw.set_channel_types(channel_types)
    
    anno_file = next((
        anno for anno in os.listdir(anno_path) 
        if subject in anno and ".mat" in anno), 
        None
    )
    # annotate raw
    add_annotation(os.path.join(anno_path, anno_file), raw)

    ### cropping data
    # check if raw data is longer than annotation
    if raw.times[-1] > raw.annotations.duration.sum():
        # crop raw to annotation
        temp_raw = crop_to_anno(raw)
    else:
        temp_raw = raw
            
    #crop the raw data
    raw_cropped = bmf.crop_data(temp_raw, wake_time)

    # ### create .mat file for the annotation
    # # get sleep states from cropped raw and save to .mat file
    # sleep_states = spf.get_stages(raw_cropped, stage_index)
    # spf.create_mat(output_path, subject, "states", sleep_states)

    ### get powerbands
    bp_mean = get_power_band(raw_cropped)

    # ### create reference channel from reference electrodes
    # # get integer indices
    # ref_indices = mne.pick_channels(raw_cropped.ch_names, ref_ch)
    # # take average of reference channels and create a new ref channel
    # reference = create_average_channel(raw_cropped, ref_indices, 'ref')
    # raw_cropped.load_data()
    # raw_cropped.add_channels([reference], force_update_info=True)

    ### get the best channel for each power band and save data in dict
    for i in range(len(target_bands)):
        # get the best channel
        best_ch = best_channel(bp_mean, target_stages[i], target_bands[i])

        # # create selection of channels    
        # ch_picks = [best_ch, "ref"]
        # # create new mne object with two channels
        # raw_picks = raw_cropped.copy().pick(ch_picks)
        # print(raw_picks.ch_names)

        raw_picks = raw_cropped.copy().pick(best_ch)
        raw_picks.load_data()

        # # process channels
        #process_channels(raw_picks, 0, 45, ch_picks)
        process_channels(raw_picks, 0, 45, best_ch)
        # # create bipolar channel
        # create_bipolar(raw_picks, ch_picks, target_bands[i])
        # # process bipolar channel
        # process_channels(raw_picks, 0, 45, "all")
        # get data from channel
        ch_data = raw_picks.get_data(best_ch)
        print(ch_data)
        # remove artefacts from data
        ch_data, _, _ = ad.removeArtefacts(ch_data[0], 250, [9,8], [0.2,0.1])
        print(len(ch_data))
        print(ch_data)

        # add channel to dictionary
        if best_ch not in final_results:
            final_results[best_ch] = [target_bands[i], ch_data]
        else:
            final_results[best_ch][0] += f"_{target_bands[i]}"

    print(final_results)

    ### save data from dict to .mat file
    for channel, ch_values in final_results.items():
        print(channel)
        print(ch_values[0])
        ch_pb = f"{channel}_" + ch_values[0]
        print(len(ch_values[1]))
        spf.create_mat(output_path, subject, ch_pb, ch_values[1])

    # ### create emg .mat file 
    # # create new mne object with two channels
    # emg_picks = raw_cropped.copy().pick(emg_ch)
    # emg_picks.load_data()
    # print(emg_picks.get_channel_types())
    # print(emg_picks.info['chs'])
    # process_channels(emg_picks, 0.5, 120, emg_ch)
    # emg_data = emg_picks.get_data()
    # l, r = emg_data
    # l, r = np.abs(l), np.abs(r)

    # # # Step 3: envelope via Hilbert
    # l = np.abs(hilbert(l))
    # r = np.abs(hilbert(r))

    # # Step 4: combine (average)
    # emg_combined = (l + r) / 2
    # spf.create_mat(output_path, subject, "EMG", emg_combined)

    # ### create eog .mat file
    # eog_picks = raw_cropped.copy().pick(eog_ch)
    # eog_picks.set_channel_types({ch: "eeg" for ch in eog_picks.ch_names})
    # print(eog_picks.ch_names)
    # process_channels(eog_picks, 0, 45, eog_ch)
    # create_bipolar(eog_picks, ['E10', 'ref'], 'eog1')
    # create_bipolar(eog_picks, ['E54', 'ref'], 'eog2')
    # create_bipolar(eog_picks, ['eog1', 'eog2'], 'eog')
    # process_channels(eog_picks, 0, 45, ['eog'])
    # eog_picks.set_channel_types({'eog': 'eog'})

    # ch_idx = eog_picks.ch_names.index('eog')
    # eog_data = eog_picks.get_data(picks=[ch_idx])
    # spf.create_mat(output_path, subject, "EOG", eog_data)



Extracting data from subject: S40_2
--------------------------------------------------
Extracting data from subject: S40_2
--------------------------------------------------
First sleep starts at: 0.0
Last sleep ends at: 30870.368
Cropping raw: 0 - 30870.368
Cropping finished.
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
['Fpz-Cz', 'Pz-Oz']
Reading 0 ... 7717592  =      0.000 ... 30870.368 secs...
Filtering raw data in 93 contiguous segments
Setting up low-pass filter at 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 75 samples (0.300 s)

[[ 1.39675658e-08 -9.76589423e-06 -1.47565362e-05 ...  2.92992981e-03
   3.13933959e-03  3.10816653e-03]]
7717593